In [ ]:
print("Training is on Google Colab.")
print("This is Regime S: a documented subsample of the official training videos.")
print("It is not a reproduction of the paper's Table 1.")
print("Quality: H.264 QP 23 (c23) only. No raw, no c40, no YouTube scrape, no generator retraining.")
print("Subsample: 10 official training video ids, the 5 pairs frozen in configs/examples/regime-s-c23-pairs.json.")
print("8 of those videos fit the network. The other 2 stay out of the optimizer and are used only for the printed validation accuracy.")
print("Official validation ids and test ids are not downloaded.")
print("Face crop uses OpenCV YuNet 2023mar, not the paper Face2Face tracker. The enlargement factor is still 1.3.")
print("Before Run all: Runtime -> Change runtime type -> T4 GPU.")
print("The training download fetches FaceForensics++ script v4 and calls it. v4 prints its own terms URL and waits for Enter.")


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/jassimarjeet/Verishield.git"
REPO = Path("/content/Verishield")
if REPO.exists():
    subprocess.check_call(["git", "-C", str(REPO), "pull", "--ff-only", "origin", "main"])
else:
    subprocess.check_call(["git", "clone", "--branch", "main", REPO_URL, str(REPO)])
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "timm"])
print("repo", REPO)


In [ ]:
import subprocess
from pathlib import Path

from src.data.ffpp_download import V1_URL, V4_URL, download_c23_subset

print("Original FaceForensics script, not used for this training download:")
print(V1_URL)
script = Path("/content/faceforensics_download_v4.py")
subprocess.check_call(["curl", "-fsSL", "-o", str(script), V4_URL])
print("Fetched FaceForensics++ v4:", script)
print("v4 prints its terms URL and waits for Enter. Press Enter at each prompt. Do not skip it.")
download_c23_subset(script, Path("/content/ffpp"), server="EU2")


In [ ]:
from pathlib import Path

from src.train.regime_s import train_regime_s

checkpoint = Path("/content/verishield-checkpoints/regime-s-c23/best.pt")
try:
    from google.colab import drive
    drive.mount("/content/drive")
    checkpoint = Path("/content/drive/MyDrive/verishield/checkpoints/regime-s-c23/best.pt")
    print("Checkpoint path is on My Drive. A Colab ephemeral disk is not the backup.")
except Exception as exc:
    print("Drive was not mounted:", exc)
    print(checkpoint)
    print("That path is on the Colab ephemeral disk. It is not the backup.")

history = train_regime_s(
    data_root=Path("/content/ffpp"),
    checkpoint_path=checkpoint,
    pretrained=True,
    seed=0,
)
print("Computed history from this session. These numbers are not filled in ahead of the run:")
for row in history:
    print(row)


In [ ]:
import os
import sys
from pathlib import Path

from google.colab import drive

REPO = Path("/content/Verishield")
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
os.chdir(REPO)

from src.eval.thresholds import (
    choose_thresholds,
    score_held_out_pair,
    sha256_file,
    write_thresholds,
)

drive.mount("/content/drive")
checkpoint = Path("/content/drive/MyDrive/verishield/checkpoints/regime-s-c23/best.pt")
if not checkpoint.is_file():
    raise FileNotFoundError(
        f"Missing {checkpoint}. Not printing a hash or a threshold."
    )
weight_sha256 = sha256_file(checkpoint)
clips = score_held_out_pair(checkpoint, Path("/content/ffpp"))
chosen = choose_thresholds(clips)
chosen["weight_sha256"] = weight_sha256
chosen["checkpoint"] = str(checkpoint)
out = Path("/content/Verishield/configs/thresholds-val.json")
write_thresholds(out, chosen, checkpoint)
print("calibration:", chosen["calibration"])
print("tau1:", chosen["tau1"])
print("tau2:", chosen["tau2"])
print(
    "validation false_accept_count:",
    chosen["false_accept_count"],
    "false_reject_count:",
    chosen["false_reject_count"],
)
print("weight_sha256:", weight_sha256)
print("wrote", out)
